In [1]:
%pip install duckdb pyarrow pandas --quiet

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/ipman/Claude Projects/Maxerience")
ACTUAL_GLOB   = str(BASE_DIR / "*/*.parquet")          # IR_Actual files (in dated subdirs)
PRODUCT_GLOB  = str(BASE_DIR / "IR_Product_*.parquet") # IR_Product files (root level)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW parquet  AS SELECT * FROM read_parquet('{ACTUAL_GLOB}')")
con.execute(f"CREATE OR REPLACE VIEW products AS SELECT * FROM read_parquet('{PRODUCT_GLOB}')")

print(f"✓ IR_Actual  → query as: parquet   ({ACTUAL_GLOB})")
print(f"✓ IR_Product → query as: products  ({PRODUCT_GLOB})")

## Inspect Schema

In [ ]:
# IR_Actual schema (reads metadata only, no full scan)
con.execute("DESCRIBE SELECT * FROM parquet").df()

## Row Counts Per File

In [ ]:
con.execute(f"""
    SELECT
        regexp_extract(filename, '[^/]+\\.parquet$') AS file,
        COUNT(*) AS row_count
    FROM read_parquet('{ACTUAL_GLOB}', filename=true)
    GROUP BY filename
    ORDER BY row_count
""").df()

## Sample Rows

In [ ]:
con.execute("SELECT * FROM parquet LIMIT 10").df()

## Query Helper

Use `query(sql)` to run any SQL. Two tables are available:

| Table | Contents | Key column |
|-------|----------|------------|
| `parquet` | IR_Actual rows (all dated files) | `ProductID` |
| `products` | IR_Product master data | `ProductId` |

**Join example:**
```sql
SELECT a.SessionUID, a.SKU, p.ProductName, p.BrandName
FROM parquet a
JOIN products p ON a.ProductID = p.ProductId
LIMIT 10
```

In [ ]:
def query(sql: str) -> pd.DataFrame:
    """Run SQL against the registered views.
    Use 'parquet' for IR_Actual data, 'products' for IR_Product data.
    """
    return con.execute(sql).df()

In [ ]:
# Query IR_Actual (unchanged)
query("SELECT count(*) FROM parquet")

In [8]:
query("SELECT count (distinct sessionUID) FROM parquet")

,count(DISTINCT sessionUID)
0,17331


In [9]:
query("SELECT count (distinct sceneUID) FROM parquet")

,count(DISTINCT sceneUID)
0,227175


In [10]:
query("SELECT count (distinct sku) FROM parquet")

,count(DISTINCT sku)
0,22708


## IR_Product Table

In [ ]:
# IR_Product schema
con.execute("DESCRIBE SELECT * FROM products").df()

In [ ]:
# Sample join: IR_Actual + IR_Product
query("""
    SELECT
        a.SessionUID,
        a.SceneUID,
        a.SKU,
        p.ProductName,
        p.BrandName,
        p.ProductCategory,
        p.BeverageType
    FROM parquet a
    JOIN products p ON a.ProductID = p.ProductId
    LIMIT 10
""")

In [11]:
query("SELECT distinct sessionUID, sceneUID, count(*) FROM parquet group by sessionuid,sceneuid order by count")

,SessionUID,SceneUID,count_star()
0,ab7ca317-4627-4933-9ccc-b5709a2be513,2a1f508e-9401-455a-8ea0-11cfffb12492,142
1,dba43478-e664-4bb6-af9f-f01255a52e28,e9cbec54-460f-44ef-a775-ab7ca48400ea,136
2,4140d8a7-e2b1-4c00-a432-7440d04c7d39,c7dbd020-8f12-40d5-bf7e-34606d3fc25d,46
3,0f310156-28f7-4e4b-9e85-8549ad0441f0,97868a81-3008-4891-a69f-933532889b05,29
4,aaa888d0-36d6-40d9-9888-bde7c246c030,5ab4970b-f8bd-431b-bd40-23ecbe5d3083,111
...,...,...,...
227170,470e7977-662b-4031-a35d-1c4fa9cb2fe3,f424cc15-a77b-469f-ace0-3e5b9319ba8e,134
227171,848d7160-e3e1-4a5f-a568-ac2338abf0b6,6663d34d-dac6-4cd3-87a9-6fe8b4cb3e32,14
227172,fbd288b5-6fe3-48b2-aff7-9aaca40eb0f1,229f5068-93e6-4fad-910a-91cb6b5311b2,16
227173,c3fb3ae6-09f6-41ce-980c-885a26dd01ec,d9856dc0-7b26-4acb-a8f9-7754c254d137,24


In [12]:
query("SELECT distinct sessionUID, sceneUID, sku, count(*) FROM parquet group by sessionuid,sceneuid,sku")

,SessionUID,SceneUID,SKU,count_star()
0,e9fe014f-e5c2-4b27-93a5-3030a95f9330,cdae5427-96e2-4383-90a8-f738be5e7c92,777515972,2
1,d4a772ea-2963-4b97-8e7c-1cf46dc62143,1323f9d1-7613-421f-8c58-26b0c39782e3,5200013516,3
2,8211aac5-2658-468b-86d3-f26091b15693,49771634-c579-4a12-87ec-451530abe709,7181700037,4
3,a2bd96e0-6325-45c5-8979-5e5dd6bddc43,fec3ddbf-cf73-4030-a959-4d8fd6e8b2cb,777454281,2
4,a2bd96e0-6325-45c5-8979-5e5dd6bddc43,77b26dd6-12dc-46d4-a0a5-1cdd8914338f,77757309,18
...,...,...,...,...
4627702,df397dca-ae47-47b7-bbbe-9f064393b97a,70d6f8b8-d36e-494b-9988-37d38e3abbef,1200000559,2
4627703,26fd8a0d-7bda-4652-8e17-4598cf039126,4b9771bc-b063-4ff8-ab3b-5449d61fc6d9,RE34365,1
4627704,f5f0c1b4-6563-4a58-9172-e5067fab3feb,93befc3f-920b-4247-9021-cf2968c087fc,RE17883,2
4627705,2c1c7c04-2950-496c-a704-336cdfcb6d88,69004780-93a5-4933-bb86-68e74efa793f,RE33918,6
